# v11 旋转检测算法验证

## 环结构
- Ring 1: r=12, 15 bits 水印
- Ring 2: r=18, 旋转检测环 (黑白交替各90°, 共轭对称4段)
- Ring 3: r=25, 45 bits 水印

## 旋转检测流程
1. 极坐标采样 r=18 附近
2. 折叠 360° → 180° (共轭对称)
3. 前缀和 + 滑动窗口找亮暗分界线
4. 分界线偏移量 = 旋转角度

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from numpy.fft import ifftshift, ifft2, fftshift, fft2

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. 生成频域模板 (水印环 + 旋转检测环)

In [ ]:
def generate_rotation_ring(L1, r_rotation, r_range, k1):
    """
    Rotation detection ring: square wave pattern (bright/dark alternating 90°)
    
    Pattern: 0°-90° bright, 90°-180° dark, 180°-270° bright (conjugate symmetric), 270°-360° dark
    
    Same embedding as watermark rings: covers r_rotation to r_rotation + r_range (2 radii)
    """
    pattern = np.zeros((L1, L1), dtype=np.float32)
    cx, cy = L1 // 2, L1 // 2
    
    # Same loop as watermark rings: r to r+r_range
    for r in range(r_rotation, r_rotation + r_range + 1):
        N = max(2, int(r * 20))
        theta_arr = np.linspace(0, 2 * np.pi, N, endpoint=False)
        
        # Square wave: 0-90 bright, 90-180 dark, 180-270 bright, 270-360 dark
        values = np.zeros(N, dtype=np.float32)
        for i, theta in enumerate(theta_arr):
            theta_deg = np.degrees(theta) % 360
            if theta_deg < 90 or (180 <= theta_deg < 270):
                values[i] = k1  # bright
            else:
                values[i] = 0    # dark
        
        x_arr = cx + np.round(r * np.cos(theta_arr)).astype(np.int32)
        y_arr = cy + np.round(r * np.sin(theta_arr)).astype(np.int32)
        
        mask = (x_arr >= 0) & (x_arr < L1) & (y_arr >= 0) & (y_arr < L1)
        pattern[y_arr[mask], x_arr[mask]] = values[mask]
        
        # Conjugate symmetric
        x_sym = (-x_arr[mask]) % L1
        y_sym = (-y_arr[mask]) % L1
        pattern[y_sym, x_sym] = values[mask]
    
    return pattern


def generate_watermark_ring(L1, r_list, bitsf, r_range, k1, numbit):
    """
    Generate watermark rings
    """
    M1 = np.zeros((L1, L1), dtype=np.float32)
    cx, cy = L1 // 2, L1 // 2
    bit_index = 0
    
    for radius_idx, r in enumerate(r_list):
        current_bits = bitsf[radius_idx]
        radius_angles = np.linspace(0, np.pi, current_bits + 1)
        
        for bit in range(current_bits):
            if numbit[bit_index + bit] == 0:
                continue
            
            bit_start = radius_angles[bit]
            bit_end = radius_angles[bit + 1]
            
            xs, ys = [], []
            for thetar in range(r, r + r_range + 1):
                N = max(2, int(thetar * 20))
                theta_arr = np.linspace(bit_start, bit_end, N)
                xs.append(cx + np.round(thetar * np.cos(theta_arr)).astype(np.int32))
                ys.append(cy + np.round(thetar * np.sin(theta_arr)).astype(np.int32))
            
            xs = np.concatenate(xs)
            ys = np.concatenate(ys)
            mask = (xs >= 0) & (xs < L1) & (ys >= 0) & (ys < L1)
            M1[ys[mask], xs[mask]] = k1
            M1[(-ys[mask]) % L1, (-xs[mask]) % L1] = k1
        
        bit_index += current_bits
    
    return M1


# Parameters
L1 = 512
k1 = 30000
r_watermark = [12, 25]
bitsf = [15, 45]
r_rotation = 18
r_range = 1
n_sectors = 60

# Random watermark bits
np.random.seed(42)
numbit = np.random.randint(0, 2, n_sectors)
print(f"Watermark bits: {numbit[:20]}... ({n_sectors} bits total)")

# Generate watermark rings
M1_watermark = generate_watermark_ring(L1, r_watermark, bitsf, r_range, k1, numbit)

# Generate rotation detection ring (square wave)
M1_rotation = generate_rotation_ring(L1, r_rotation, r_range, k1)

# Combine
M1_total = M1_watermark + M1_rotation

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(M1_watermark, cmap='hot')
axes[0].set_title('Watermark Rings (r=12, 25)')
axes[1].imshow(M1_rotation, cmap='hot')
axes[1].set_title(f'Rotation Ring (r={r_rotation}, Square 90°)')
axes[2].imshow(M1_total, cmap='hot')
axes[2].set_title('Combined Template')
plt.tight_layout()
plt.show()

# Verify ring coverage
cx, cy = L1 // 2, L1 // 2
theta_check = np.linspace(0, 2 * np.pi, 360, endpoint=False)
print(f"\nRing coverage verification:")
for r in [12, 13, 18, 19, 25, 26]:
    x_arr = cx + np.round(r * np.cos(theta_check)).astype(np.int32)
    y_arr = cy + np.round(r * np.sin(theta_check)).astype(np.int32)
    mask = (x_arr >= 0) & (x_arr < L1) & (y_arr >= 0) & (y_arr < L1)
    values = M1_total[y_arr[mask], x_arr[mask]]
    nonzero = np.count_nonzero(values)
    print(f"  r={r}: nonzero={nonzero}/{len(values)}, mean={values.mean():.0f}")

## 2. 生成空域模板并嵌入

In [ ]:
# IFFT -> spatial domain
spatial = np.real(ifft2(ifftshift(M1_total)))
Tm = np.where(spatial < 0, 0, 255).astype(np.uint8)

# Read test image
import os
img_dir = '/mnt/xsj2023/Datasets/COCO/coco_minator_dataset/train'
img_files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
host = cv2.imread(os.path.join(img_dir, img_files[0]))
host = cv2.resize(host, (L1, L1))

# Cb channel embedding (alpha blending)
alpha = 0.016
ycrcb = cv2.cvtColor(host, cv2.COLOR_BGR2YCrCb).astype(np.float32)
y_ch, cr_ch, cb_ch = cv2.split(ycrcb)
Tm_gray = Tm.astype(np.float32)
cb_wm = cb_ch * (1 - alpha) + Tm_gray * alpha
cb_wm = np.clip(cb_wm, 0, 255).astype(np.uint8)
ycrcb_wm = cv2.merge([y_ch.astype(np.uint8), cr_ch.astype(np.uint8), cb_wm])
watermarked = cv2.cvtColor(ycrcb_wm, cv2.COLOR_YCrCb2BGR)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cv2.cvtColor(host, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image')
axes[1].imshow(Tm, cmap='gray')
axes[1].set_title('Spatial Template Tm')
axes[2].imshow(cv2.cvtColor(watermarked, cv2.COLOR_BGR2RGB))
axes[2].set_title('Watermarked Image')
plt.tight_layout()
plt.show()

## 3. FFT + 极坐标采样

In [ ]:
def cartesian_to_polar(fft_mag, r_min, r_max, angle_bins):
    """
    Polar coordinate sampling (full 360°)
    
    Args:
        fft_mag: (H, W) FFT magnitude spectrum
        r_min, r_max: sampling radius range
        angle_bins: number of angle samples (360 = 1 per degree)
    
    Returns:
        polar: (r_bins, angle_bins) polar representation
    """
    H, W = fft_mag.shape
    cx, cy = H // 2, W // 2
    
    r_bins = r_max - r_min + 1
    polar = np.zeros((r_bins, angle_bins), dtype=np.float64)
    
    # Sample full 360° (0 to 2π)
    theta_arr = np.linspace(0, 2 * np.pi, angle_bins, endpoint=False)
    
    for ri in range(r_bins):
        r = r_min + ri
        x_arr = cx + np.round(r * np.cos(theta_arr)).astype(np.int32)
        y_arr = cy + np.round(r * np.sin(theta_arr)).astype(np.int32)
        
        mask = (x_arr >= 0) & (x_arr < W) & (y_arr >= 0) & (y_arr < H)
        polar[ri, mask] = fft_mag[y_arr[mask], x_arr[mask]]
    
    return polar


# FFT
gray = cv2.cvtColor(watermarked, cv2.COLOR_BGR2GRAY).astype(np.float32)
fft_result = fftshift(fft2(gray))
fft_mag = np.abs(fft_result)

# Polar sampling near rotation ring
rotation_sample_width = 3  # Sampling width: r_rotation ± width
r_min = r_rotation - rotation_sample_width
r_max = r_rotation + rotation_sample_width
angle_bins = 360

polar_rotation = cartesian_to_polar(fft_mag, r_min, r_max, angle_bins)

print(f"Rotation ring polar sampling: r=[{r_min}, {r_max}], angle_bins={angle_bins}")
print(f"polar shape: {polar_rotation.shape}")

# Average along radius to get 1D signal
signal_360 = polar_rotation.mean(axis=0)  # (360,)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(np.log1p(fft_mag), cmap='hot')
axes[0].set_title('FFT Magnitude (log)')
axes[1].plot(signal_360)
axes[1].set_title(f'Rotation Ring Signal (r={r_rotation}±{rotation_sample_width}), 360°')
axes[1].set_xlabel('Angle (°)')
axes[1].axvline(x=90, color='r', linestyle='--', alpha=0.5, label='90°')
axes[1].axvline(x=180, color='g', linestyle='--', alpha=0.5, label='180°')
axes[1].axvline(x=270, color='r', linestyle='--', alpha=0.5, label='270°')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Template FFT visualization
# 2. 3x3 tile + rotate + crop
# 3. Polar sampling of three rings

cx, cy = L1 // 2, L1 // 2

# === 1. Template FFT ===
fft_template = fftshift(fft2(Tm.astype(np.float32)))
mag_template = np.abs(fft_template)

# Individual ring FFT
fft_wm = fftshift(fft2(np.where(np.real(ifft2(ifftshift(M1_watermark))) < 0, 0, 255).astype(np.float32)))
fft_rot = fftshift(fft2(np.where(np.real(ifft2(ifftshift(M1_rotation))) < 0, 0, 255).astype(np.float32)))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(np.log1p(mag_template), cmap='hot')
axes[0].set_title('Combined Template FFT (log)')
axes[0].plot([cx, cx], [0, L1-1], 'c--', alpha=0.3)
axes[0].plot([0, L1-1], [cy, cy], 'c--', alpha=0.3)

axes[1].imshow(np.log1p(np.abs(fft_wm)), cmap='hot')
axes[1].set_title('Watermark Ring Template FFT')

axes[2].imshow(np.log1p(np.abs(fft_rot)), cmap='hot')
axes[2].set_title('Rotation Ring Template FFT')

axes[3].imshow(np.log1p(mag_template), cmap='hot')
axes[3].set_title('Three Rings Labeled')
for r, color, name in [(12, 'lime', 'r=12'), (18, 'cyan', 'r=18'), (25, 'yellow', 'r=25')]:
    circle = plt.Circle((cx, cy), r, fill=False, color=color, linewidth=1.5)
    axes[3].add_patch(circle)
    axes[3].text(cx + r + 2, cy - 2, name, color=color, fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()


# === 2. 3x3 Tile + Rotate + Crop ===
def tile3x3_rotate_crop(image, angle):
    """3x3 tile -> rotate -> center crop 1x1"""
    h, w = image.shape[:2]
    if len(image.shape) == 2:
        tiled = np.tile(image, (3, 3))
    else:
        tiled = np.tile(image, (3, 3, 1))
    center_tiled = (tiled.shape[1] // 2, tiled.shape[0] // 2)
    M = cv2.getRotationMatrix2D(center_tiled, angle, 1.0)
    rotated = cv2.warpAffine(tiled, M, (tiled.shape[1], tiled.shape[0]),
                              borderMode=cv2.BORDER_WRAP)
    crop_x = w
    crop_y = h
    cropped = rotated[crop_y:crop_y+h, crop_x:crop_x+w]
    return cropped


# Cb channel rotation test
ycrcb_out = cv2.cvtColor(watermarked, cv2.COLOR_BGR2YCrCb)
cb_orig = ycrcb_out[:, :, 2].astype(np.float32)

cb_rot5 = tile3x3_rotate_crop(cb_orig, 5)
cb_rot10 = tile3x3_rotate_crop(cb_orig, 10)

# FFT
fft_orig = fftshift(fft2(cb_orig))
fft_rot5 = fftshift(fft2(cb_rot5))
fft_rot10 = fftshift(fft2(cb_rot10))

mag_orig = np.abs(fft_orig)
mag_rot5 = np.abs(fft_rot5)
mag_rot10 = np.abs(fft_rot10)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes[0, 0].imshow(cb_orig, cmap='gray')
axes[0, 0].set_title('Cb Channel (No Rotation)')
axes[0, 1].imshow(cb_rot5, cmap='gray')
axes[0, 1].set_title('Cb Channel (Rotate 5°)')
axes[0, 2].imshow(cb_rot10, cmap='gray')
axes[0, 2].set_title('Cb Channel (Rotate 10°)')

axes[1, 0].imshow(np.log1p(mag_orig), cmap='hot')
axes[1, 0].set_title('FFT (No Rotation)')
axes[1, 1].imshow(np.log1p(mag_rot5), cmap='hot')
axes[1, 1].set_title('FFT (Rotate 5°)')
axes[1, 2].imshow(np.log1p(mag_rot10), cmap='hot')
axes[1, 2].set_title('FFT (Rotate 10°)')

for ax in axes[1]:
    for r in [12, 18, 25]:
        circle = plt.Circle((cx, cy), r, fill=False, color='cyan', linewidth=1, linestyle='--')
        ax.add_patch(circle)
plt.tight_layout()
plt.show()


# === 3. Polar Sampling of Three Rings ===
def sample_ring_signal(mag, r, n_angles=360):
    """Sample 360° signal at specified radius"""
    theta_arr = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    x_arr = cx + np.round(r * np.cos(theta_arr)).astype(np.int32)
    y_arr = cy + np.round(r * np.sin(theta_arr)).astype(np.int32)
    mask = (x_arr >= 0) & (x_arr < L1) & (y_arr >= 0) & (y_arr < L1)
    signal = np.zeros(n_angles)
    signal[mask] = mag[y_arr[mask], x_arr[mask]]
    return signal


ring_configs = [
    (12, 'Watermark Ring 1 (r=12)'),
    (18, 'Rotation Ring (r=18)'),
    (25, 'Watermark Ring 2 (r=25)')
]

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
for row, (r, name) in enumerate(ring_configs):
    sig_orig = sample_ring_signal(mag_orig, r)
    sig_rot5 = sample_ring_signal(mag_rot5, r)
    sig_rot10 = sample_ring_signal(mag_rot10, r)
    
    axes[row].plot(sig_orig, label='No Rotation', alpha=0.8)
    axes[row].plot(sig_rot5, label='Rotate 5°', alpha=0.8)
    axes[row].plot(sig_rot10, label='Rotate 10°', alpha=0.8)
    axes[row].set_title(f'{name} Polar Sampling Signal')
    axes[row].set_xlabel('Angle (°)')
    axes[row].legend()
    
    print(f"{name}:")
    print(f"  No Rotation: mean={sig_orig.mean():.1f}, std={sig_orig.std():.1f}")
    print(f"  Rotate 5°:   mean={sig_rot5.mean():.1f}, std={sig_rot5.std():.1f}")
    print(f"  Rotate 10°:  mean={sig_rot10.mean():.1f}, std={sig_rot10.std():.1f}")
    print()

plt.tight_layout()
plt.show()

In [ ]:
# Fold: s_fold[θ] = s[θ] + s[θ+180]
signal_folded = signal_360[:180] + signal_360[180:360]

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(signal_360)
axes[0].set_title('Original Signal (360°)')
axes[0].set_xlabel('Angle (°)')
axes[0].axvline(x=180, color='r', linestyle='--', alpha=0.5)
axes[0].legend(['Signal', '180° boundary'])

axes[1].plot(signal_folded)
axes[1].set_title('Folded Signal (180°)')
axes[1].set_xlabel('Angle (°)')
axes[1].axvline(x=90, color='r', linestyle='--', alpha=0.5, label='Expected boundary (90°)')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Folded signal length: {len(signal_folded)}")
print(f"0-90° mean: {signal_folded[:90].mean():.2f} (should be bright)")
print(f"90-180° mean: {signal_folded[90:].mean():.2f} (should be dark)")

## 5. 前缀和 + 滑动窗口找分界线

对每个角度θ, 计算左90°求和 vs 右90°求和的差值
用前缀和实现O(1)查询

In [ ]:
def find_boundary_by_prefix_sum(signal, half_window):
    """
    Find bright-dark boundary using prefix sum on circular array (square wave detection)
    
    Args:
        signal: 1D signal (length N, representing 180°)
        half_window: each side window size (90° corresponds to 90 points)
    
    Returns:
        diff: difference array
        boundary_idx: boundary position
    """
    N = len(signal)
    
    # Prefix sum (extend to 3x to ensure no out-of-bounds)
    signal_ext = np.concatenate([signal, signal, signal])
    cumsum = np.zeros(len(signal_ext) + 1)
    cumsum[1:] = np.cumsum(signal_ext)
    
    # Calculate left and right sums for each position
    diff = np.zeros(N)
    for i in range(N):
        idx = i + N
        
        # Left half: [idx - half_window, idx)
        left_sum = cumsum[idx] - cumsum[idx - half_window]
        
        # Right half: [idx, idx + half_window)
        right_sum = cumsum[idx + half_window] - cumsum[idx]
        
        diff[i] = left_sum - right_sum
    
    # Find position with maximum absolute difference
    boundary_idx = np.argmax(np.abs(diff))
    
    return diff, boundary_idx


def detect_rotation_full(watermarked_bgr, r_rotation, sample_width, half_window=90):
    """
    Complete rotation detection flow (square wave + prefix sum)
    
    Args:
        watermarked_bgr: BGR watermarked image
        r_rotation: rotation ring radius
        sample_width: sampling width
        half_window: each side window size (90°)
    
    Returns:
        rotation_deg: detected rotation angle
        boundary_idx: boundary position
    """
    # Extract Cb channel
    ycrcb = cv2.cvtColor(watermarked_bgr, cv2.COLOR_BGR2YCrCb)
    cb = ycrcb[:, :, 2].astype(np.float32)
    
    # FFT
    fft_cb = fftshift(fft2(cb))
    mag_cb = np.abs(fft_cb)
    
    # Polar sampling (full 360°)
    r_min = r_rotation - sample_width
    r_max = r_rotation + sample_width
    polar = cartesian_to_polar(mag_cb, r_min, r_max, 360)
    
    # Average along radius
    signal_360 = polar.mean(axis=0)
    
    # Fold 360° -> 180° (conjugate symmetric)
    signal_folded = signal_360[:180] + signal_360[180:360]
    
    # Prefix sum to find boundary
    diff, boundary_idx = find_boundary_by_prefix_sum(signal_folded, half_window)
    
    # Calculate rotation angle (boundary expected at 90°)
    rotation_deg = boundary_idx - 90
    
    # Handle ±180° range
    if rotation_deg > 90:
        rotation_deg -= 180
    elif rotation_deg < -90:
        rotation_deg += 180
    
    return rotation_deg, boundary_idx


# Test no rotation case
rotation_deg, boundary_idx = detect_rotation_full(watermarked, r_rotation, sample_width=3)

print(f"No rotation detection result: {rotation_deg:.1f}°")
print(f"Boundary position: {boundary_idx}°")
print(f"Expected: 90°")

# Visualization
ycrcb_out = cv2.cvtColor(watermarked, cv2.COLOR_BGR2YCrCb)
cb_out = ycrcb_out[:, :, 2].astype(np.float32)
fft_cb = fftshift(fft2(cb_out))
mag_cb = np.abs(fft_cb)

polar = cartesian_to_polar(mag_cb, r_rotation - 3, r_rotation + 3, 360)
signal_360 = polar.mean(axis=0)
signal_folded = signal_360[:180] + signal_360[180:360]
diff, _ = find_boundary_by_prefix_sum(signal_folded, 90)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
axes[0].plot(signal_360)
axes[0].set_title('Rotation Ring Signal (Cb FFT, r=18±3, 360°)')
axes[0].set_xlabel('Angle (°)')

axes[1].plot(signal_folded)
axes[1].axvline(x=boundary_idx, color='r', linestyle='--', label=f'Boundary ({boundary_idx}°)')
axes[1].set_title('Folded Signal (180°)')
axes[1].set_xlabel('Angle (°)')
axes[1].legend()

axes[2].plot(diff)
axes[2].axvline(x=boundary_idx, color='r', linestyle='--', label=f'Max Difference ({boundary_idx}°)')
axes[2].set_title('Left-Right 90° Sum Difference')
axes[2].set_xlabel('Angle (°)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. 验证: 加旋转后重新检测

In [ ]:
# Test different rotation angles
test_angles = [0, 3, -3, 5, -5, 10, -10]

print("Rotation Detection Test (Square Wave + Prefix Sum):")
print("-" * 50)
for angle in test_angles:
    center = (L1 // 2, L1 // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(watermarked, M, (L1, L1))
    
    detected, boundary_idx = detect_rotation_full(rotated, r_rotation, sample_width=3)
    error = abs(detected - angle)
    
    status = "OK" if error < 2 else "FAIL"
    print(f"Actual: {angle:+3.0f}° | Detected: {detected:+6.1f}° | Boundary: {boundary_idx:3d}° | Error: {error:.1f}° {status}")

## 7. 采样宽度对比

In [ ]:
# Test different sampling widths
sample_widths = [1, 2, 3, 5]
test_angle = 5  # Fixed test angle

center = (L1 // 2, L1 // 2)
M = cv2.getRotationMatrix2D(center, test_angle, 1.0)
rotated = cv2.warpAffine(watermarked, M, (L1, L1))

print(f"Fixed rotation angle: {test_angle}°")
print(f"Detection results for different sampling widths:")
print("-" * 40)

for sw in sample_widths:
    detected, boundary_idx = detect_rotation_full(rotated, r_rotation, sample_width=sw)
    print(f"  width={sw}: detected={detected:+6.1f}°, boundary={boundary_idx}°")

## 8. 完整流水线可视化

In [ ]:
# Full pipeline visualization with 5° rotation
test_angle = 5
center = (L1 // 2, L1 // 2)
M = cv2.getRotationMatrix2D(center, test_angle, 1.0)
rotated = cv2.warpAffine(watermarked, M, (L1, L1))

# Detect rotation
detected, boundary_idx = detect_rotation_full(rotated, r_rotation, sample_width=3)

# Get intermediate results for visualization
ycrcb_rot = cv2.cvtColor(rotated, cv2.COLOR_BGR2YCrCb)
cb_rot = ycrcb_rot[:, :, 2].astype(np.float32)
fft_rot = fftshift(fft2(cb_rot))
mag_rot = np.abs(fft_rot)

sw = 3
polar = cartesian_to_polar(mag_rot, r_rotation - sw, r_rotation + sw, 360)
signal_360 = polar.mean(axis=0)
signal_folded = signal_360[:180] + signal_360[180:360]
diff, _ = find_boundary_by_prefix_sum(signal_folded, 90)

# Visualization
fig, axes = plt.subplots(3, 2, figsize=(14, 12))

# Row 1: Original vs Rotated
axes[0, 0].imshow(cv2.cvtColor(watermarked, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('Original Watermarked Image')
axes[0, 1].imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title(f'Rotated {test_angle}°')

# Row 2: FFT + 360° signal
axes[1, 0].imshow(np.log1p(mag_rot), cmap='hot')
axes[1, 0].set_title('Cb Channel FFT Magnitude')
axes[1, 1].plot(signal_360)
axes[1, 1].set_title(f'Rotation Ring Signal (r={r_rotation}±{sw}, 360°)')
axes[1, 1].set_xlabel('Angle (°)')

# Row 3: Folded signal + difference curve
axes[2, 0].plot(signal_folded)
axes[2, 0].axvline(x=boundary_idx, color='r', linestyle='--', label=f'Boundary ({boundary_idx}°)')
axes[2, 0].set_title('Folded Signal (180°)')
axes[2, 0].set_xlabel('Angle (°)')
axes[2, 0].legend()

axes[2, 1].plot(diff)
axes[2, 1].axvline(x=boundary_idx, color='r', linestyle='--', label=f'Boundary ({boundary_idx}°)')
axes[2, 1].set_title(f'Left-Right 90° Sum Diff, Detected={detected:.1f}°')
axes[2, 1].set_xlabel('Angle (°)')
axes[2, 1].legend()

plt.tight_layout()
plt.show()

print(f"\nResult: Actual={test_angle}°, Detected={detected:.1f}°, Boundary={boundary_idx}°")

## 9. 总结

### 旋转环设计
- **模式**: 方波 (黑白交替各90°)
- **结构**: 0°-90°亮, 90°-180°暗, 180°-270°亮(共轭对称), 270°-360°暗(共轭对称)
- **值域**: 0 或 k1

### 检测算法
1. **极坐标采样**: 在r=18±width范围内采样完整360°
2. **沿半径取均值**: 得到1D信号
3. **折叠**: 360° → 180° (利用共轭对称)
4. **前缀和**: 计算cumsum, 支持O(1)的区间求和
5. **滑动窗口**: 对每个θ, 计算左90°和右90°的求和差
6. **找最大值**: |diff|最大的位置就是亮暗分界线
7. **计算角度**: 分界线位置 - 90° = 旋转角度

### 参数
- `r_rotation = 18`: 旋转环半径
- `sample_width`: 采样宽度, 推荐3
- `half_window = 90`: 每侧窗口90°
- `angle_bins = 360`: 角度分辨率1°

### 与原始v11的区别
| | 原始v11 | 新设计 |
|---|---|---|
| 旋转环模式 | cos(8θ), 45°周期 | 方波, 90°亮暗交替 |
| 检测方法 | 互相关 | 前缀和+滑动窗口 |
| 检测精度 | ±0.5° (±5°范围) | 待验证 |